**Finetune MoLFormer**

In [ ]:
import torch
import pandas as pd
import numpy as np
from rdkit import Chem
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW, lr_scheduler
from torch import nn
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)
BATCH_SIZE = 64
MAX_LEN = 256
EPOCHS = 40
CHEMBERT_MODEL_NAME = "ibm/MoLFormer-XL-both-10pct"
SAVE_PATH = "finetuned_molformer"
LEARNING_RATE = 4e-5
WEIGHT_DECAY = 0.001
GRAD_CLIP = 1.0
df = pd.read_csv("df.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
tokenizer = AutoTokenizer.from_pretrained(CHEMBERT_MODEL_NAME, trust_remote_code=True)


class MPDataset(Dataset):
    def __init__(self, smiles, mp):
        self.smiles = smiles
        self.mp = mp

    def __len__(self):
        return len(self.mp)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.smiles[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "MP": torch.FloatTensor([self.mp[idx]]),
        }


scaler = StandardScaler()
scaler.fit(train_df["MP"].values.reshape(-1, 1))
y_train = scaler.transform(train_df["MP"].values.reshape(-1, 1))
y_val = scaler.transform(val_df["MP"].values.reshape(-1, 1))

train_dataset = MPDataset(train_df["SMILES"].values, y_train.flatten())
val_dataset = MPDataset(val_df["SMILES"].values, y_val.flatten())
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True
)


class MPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.chemberta = AutoModel.from_pretrained(
            CHEMBERT_MODEL_NAME, trust_remote_code=True
        )
        self.regressor = nn.Sequential(
            nn.Linear(self.chemberta.config.hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )
        for param in self.chemberta.encoder.layer[:5].parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.chemberta(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        pooled_output = (hidden_states * attention_mask.unsqueeze(-1)).sum(
            1
        ) / attention_mask.sum(1, keepdim=True)
        return self.regressor(pooled_output)


model = MPModel().to(DEVICE)
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=len(train_loader) * 3, eta_min=LEARNING_RATE / 10
)
loss_fn = nn.HuberLoss()


def compute_metrics(outputs, targets):
    mae = mean_absolute_error(targets, outputs)
    r2 = r2_score(targets, outputs)
    return {"mae": mae, "r2": r2}


best_val_mae = float("inf")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["MP"].to(DEVICE)
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
    model.eval()
    val_loss = 0
    all_outputs = []
    all_targets = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            targets = batch["MP"].to(DEVICE)
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, targets)
            val_loss += loss.item()
            all_outputs.extend(outputs.cpu().numpy().flatten())
            all_targets.extend(targets.cpu().numpy().flatten())
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    metrics = compute_metrics(all_outputs, all_targets)
    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"MAE: {metrics['mae']:.4f} | R²: {metrics['r2']:.4f}")
    if metrics["mae"] < best_val_mae:
        best_val_mae = metrics["mae"]
        torch.save(model.state_dict(), f"{SAVE_PATH}_best.pt")
        print(f"Saved best model with MAE: {best_val_mae:.4f}")
torch.save(model.state_dict(), f"{SAVE_PATH}_final.pt")
print("\nTraining completed!")
print(f"Best model saved to: {SAVE_PATH}_best.pt")


In [2]:
print(f"\nEpoch {epoch+1}")
print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
print(f"MAE: {metrics['mae']:.4f} | R²: {metrics['r2']:.4f}")


Epoch 40
Train Loss: 0.0406 | Val Loss: 0.1270
MAE: 0.3851 | R²: 0.7314


In [3]:
sigma = scaler.scale_ 
MAE_original = metrics['mae']:.4f} * sigma
MAE_original

array([27.40870781])